[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/projects/06_fair-credit-scoring.ipynb)

*Industry Project 6 of 8 · Data Scientist · Advanced · about 8–10 hours*

# Fair Credit-Risk Scoring

**The situation.** *QuickCredit*, a digital lender, approves personal loans in minutes. Regulators and the company's own ethics board require that the automated decision is **accurate, well-calibrated, fair across groups, and explainable** to every applicant who is turned down.

The model must not use gender. But removing a column doesn't guarantee fairness: other features can carry the same information. You'll build the model, set the approval policy, then audit it like a model-risk reviewer would.

**What you'll hand in:** a default-risk model with calibration evidence, an approval policy, a fairness audit (selection rates, disparate impact, equal opportunity), reason codes for rejections, and a counterfactual for one applicant.

**Skills:** logistic regression, calibration, Brier score, approval policy, fairness metrics, disparate impact, reason codes, counterfactuals

**How this works:** each milestone has a task, a code cell for you to finish, and a ✅ check. Hints and a reference solution are hidden until you open them. Try for at least 20 minutes before opening the solution — the struggle is where the learning happens.

## The data

**credit_applicants.csv** — 6,000 past applicants with known outcomes:

| Column | Meaning |
|---|---|
| applicant_id | ID |
| age, gender, region | demographics (`gender` must **not** be a model input; it's kept only for the audit) |
| income_k | monthly income, ₹ thousand |
| employment_years | years in current employment |
| debt_to_income | existing debt payments ÷ income |
| prior_defaults | number of past defaults |
| loan_amount_k | requested amount, ₹ thousand |
| credit_history_years | length of credit history |
| default | **target**: 1 if the loan defaulted |

In [ ]:
# ▶ Run this cell first: it loads the libraries, the data and the checker.
import math, os, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/abpandey-dev/learn-python-from-scratch-to-advanced/main/projects/data/"


def load(name):
    """Read a project data file: from the local data/ folder if it is there, otherwise from GitHub."""
    local = Path("data") / name
    return pd.read_csv(local if local.exists() else RAW + name)


def _plain(x):
    if isinstance(x, pd.DataFrame):
        return {str(c): _plain(x[c].tolist()) for c in x.columns}
    if isinstance(x, pd.Series):
        return {_plain(k): _plain(v) for k, v in x.items()}
    if isinstance(x, np.ndarray):
        return _plain(x.tolist())
    if isinstance(x, np.generic):
        return _plain(x.item())
    if isinstance(x, (list, tuple)):
        return [_plain(v) for v in x]
    if isinstance(x, dict):
        return {_plain(k): _plain(v) for k, v in x.items()}
    return x


def _same(a, b, tol):
    if isinstance(b, bool) or isinstance(a, bool):
        return a == b
    if isinstance(b, (int, float)) and isinstance(a, (int, float)):
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol if tol > 1e-3 else 1e-6)
    if isinstance(b, list) and isinstance(a, list):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(b, dict) and isinstance(a, dict):
        return set(map(str, a)) == set(map(str, b)) and all(_same(a[k], b[k], tol) for k in a if k in b)
    return a == b


def _why(a, b):
    if a is None:
        return "your function returned None — finish the code and return the result."
    if type(a) != type(b) and not (isinstance(a, (int, float)) and isinstance(b, (int, float))):
        return f"expected a {type(b).__name__}, got a {type(a).__name__}."
    if isinstance(b, (list, dict)) and len(a) != len(b):
        return f"expected {len(b)} items, got {len(a)}."
    if isinstance(b, dict) and set(map(str, a)) != set(map(str, b)):
        return f"the keys should be {sorted(map(str, b))[:8]}."
    return "the values are not right yet. Re-read the task and the hints."


def check(name, fn, want, tol=1e-3):
    """Run your code for one milestone and compare it with the expected answer."""
    try:
        got = _plain(fn())
    except NotImplementedError:
        print(f"⬜ {name}: not started yet.")
        return False
    except Exception as e:
        print(f"❌ {name}: your code raised {type(e).__name__}: {e}")
        return False
    if _same(got, want, tol):
        print(f"✅ {name}: correct!")
        return True
    print(f"❌ {name}: not yet — {_why(got, want)}")
    return False

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
apps = load("credit_applicants.csv")
NUM = ["age", "income_k", "employment_years", "debt_to_income", "prior_defaults", "loan_amount_k", "credit_history_years"]
CAT = ["region"]
FEATURES = NUM + CAT
CUTOFF = 0.15        # approve when predicted default probability is below this
train, test = train_test_split(apps, test_size=0.3, stratify=apps["default"], random_state=7)


# Used by the checks: your model's test-set probabilities.
def test_pd():
    model = fit_model(train)
    return model, model.predict_proba(test[FEATURES])[:, 1]
print("Ready! Data loaded.")

## Milestone 1: Know the base rates

Write `default_rates(apps)` returning `{"overall": %, "F": %, "M": %, "urban": %, "rural": %}` — default rates rounded to 2.

Differences in base rates between groups are a fact about the data; how the model treats people *given* their risk is what the fairness audit will check.

In [ ]:
def default_rates(apps):
    raise NotImplementedError

In [ ]:
check("Know the base rates", lambda: default_rates(apps), {'overall': 10.15, 'F': 10.43, 'M': 9.95, 'rural': 12.8, 'urban': 8.77}, tol=0.001)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def default_rates(apps):
    out = {"overall": round(apps["default"].mean() * 100, 2)}
    for col in ["gender", "region"]:
        out.update((apps.groupby(col)["default"].mean() * 100).round(2).to_dict())
    return out
```

</details>

## Milestone 2: An explainable risk model

Write `fit_model(train)` returning a fitted `Pipeline`: one-hot `CAT`, standard-scale `NUM`, then `LogisticRegression(max_iter=1000)`, trained on `FEATURES` (no gender!) → `default`.

The check reports test ROC AUC. Lenders often prefer logistic regression: every coefficient can be explained and audited.

In [ ]:
def fit_model(train):
    raise NotImplementedError

In [ ]:
check("An explainable risk model", lambda: round(roc_auc_score(test['default'], test_pd()[1]), 3), 0.772, tol=0.01)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def fit_model(train):
    prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT), ("num", StandardScaler(), NUM)])
    model = Pipeline([("prep", prep), ("lr", LogisticRegression(max_iter=1000))])
    return model.fit(train[FEATURES], train["default"])
```

</details>

## Milestone 3: Can we trust the probabilities?

Pricing and the cut-off use the probability itself, so it must be **calibrated**: among applicants scored 10%, about 10% should default.

Write `calibration(y, p, bins=10)` returning `[Brier score, ECE]` rounded to 4. ECE = Σ over non-empty equal-width bins of (bin share of rows) × |mean predicted − observed default rate|.

<details>
<summary><b>💡 Hint</b></summary>

Bin index = `np.minimum((p * bins).astype(int), bins - 1)`; then group by it.

</details>

In [ ]:
def calibration(y, p, bins=10):
    raise NotImplementedError

In [ ]:
check("Can we trust the probabilities?", lambda: calibration(test['default'], test_pd()[1]), [0.0821, 0.0189], tol=0.05)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def calibration(y, p, bins=10):
    y, p = np.asarray(y), np.asarray(p)
    b = np.minimum((p * bins).astype(int), bins - 1)
    ece = sum((b == k).mean() * abs(p[b == k].mean() - y[b == k].mean()) for k in np.unique(b))
    return [round(float(brier_score_loss(y, p)), 4), round(float(ece), 4)]
```

</details>

## Milestone 4: The approval policy and its impact on groups

Approve when predicted default probability < `CUTOFF` (0.15). Write `approval_audit(test, p)` returning:
`{"approval_rate": %, "F": %, "M": %, "disparate_impact": lower group rate ÷ higher group rate}` — rates rounded to 1, the ratio to 3.

The **four-fifths rule** flags a ratio below 0.8 for review.

In [ ]:
def approval_audit(test, p):
    raise NotImplementedError

In [ ]:
check("The approval policy and its impact on groups", lambda: approval_audit(test, test_pd()[1]), {'approval_rate': 78.7, 'F': 77.4, 'M': 79.6, 'disparate_impact': 0.972}, tol=0.03)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def approval_audit(test, p):
    approved = pd.Series(p < CUTOFF, index=test.index)
    by = approved.groupby(test["gender"]).mean() * 100
    return {"approval_rate": round(approved.mean() * 100, 1), "F": round(by["F"], 1), "M": round(by["M"], 1),
            "disparate_impact": round(by.min() / by.max(), 3)}
```

</details>

## Milestone 5: Equal opportunity for good borrowers

Different approval rates can come from different risk levels. A sharper question: **among applicants who would have repaid** (`default == 0`), are both genders approved at the same rate?

Write `equal_opportunity(test, p)` returning `{"F": %, "M": %, "gap_pp": M − F}`, rounded to 1.

In [ ]:
def equal_opportunity(test, p):
    raise NotImplementedError

In [ ]:
check("Equal opportunity for good borrowers", lambda: equal_opportunity(test, test_pd()[1]), {'F': 80.6, 'M': 83.0, 'gap_pp': 2.4}, tol=0.04)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def equal_opportunity(test, p):
    good = test["default"] == 0
    approved = pd.Series(p < CUTOFF, index=test.index)
    rate = approved[good].groupby(test.loc[good, "gender"]).mean() * 100
    return {"F": round(rate["F"], 1), "M": round(rate["M"], 1), "gap_pp": round(rate["M"] - rate["F"], 1)}
```

</details>

The model never saw gender, yet good borrowers of the two groups are approved at slightly different rates (a gap of a few points), because **income** — a model input — differs between groups in this data. That's the proxy problem, here in a mild form. A ratio of 0.97 passes the four-fifths rule, but an auditor will still ask you to explain the gap. Options: accept it if income is a legitimate, necessary risk factor; adjust the policy; or collect better features. Write down which you'd recommend and why — that judgement is the point of the audit.

## Milestone 6: Reason codes for rejected applicants

Lenders must tell rejected applicants **why**. For a logistic model, each scaled numeric feature contributes `coefficient × scaled value` to the risk score.

Write `reasons(model, row, n=2)` returning the names of the n numeric features (from `NUM`) pushing this applicant's risk **up** the most, biggest first. The check runs it on the first three rejected test applicants.

<details>
<summary><b>💡 Hint</b></summary>

`prep = model.named_steps['prep']`; `x = prep.transform(row[FEATURES])[0]`; names from `prep.get_feature_names_out()` look like `num__income_k`.

</details>

In [ ]:
def reasons(model, row, n=2):
    raise NotImplementedError

In [ ]:
check("Reason codes for rejected applicants", lambda: [reasons(test_pd()[0], test.iloc[[i]]) for i in np.flatnonzero(test_pd()[1] >= CUTOFF)[:3]], [['prior_defaults', 'income_k'], ['debt_to_income', 'prior_defaults'], ['debt_to_income', 'prior_defaults']], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def reasons(model, row, n=2):
    prep, lr = model.named_steps["prep"], model.named_steps["lr"]
    x = prep.transform(row[FEATURES])[0]
    names = prep.get_feature_names_out()
    contrib = {name.split("__")[1]: c * v for name, c, v in zip(names, lr.coef_[0], x) if name.startswith("num__")}
    return sorted(contrib, key=contrib.get, reverse=True)[:n]
```

</details>

## Milestone 7: What would get them approved?

Write `income_needed(model, row)` returning the smallest monthly income (₹ thousand, a whole number) at which this applicant's predicted default probability drops below `CUTOFF`, keeping everything else the same — or `None` if no income up to 1,000 works. The check uses the first rejected test applicant.

<details>
<summary><b>💡 Hint</b></summary>

Copy the row, try income = current, current + 1, … and predict each time. (A smarter way: solve the logistic equation directly — try it after.)

</details>

In [ ]:
def income_needed(model, row):
    raise NotImplementedError

In [ ]:
check("What would get them approved?", lambda: income_needed(test_pd()[0], test.iloc[[int(np.flatnonzero(test_pd()[1] >= CUTOFF)[0])]]), 58, tol=0.05)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def income_needed(model, row):
    trial = row[FEATURES].copy()
    for income in range(int(math.ceil(row["income_k"].iloc[0])), 1001):
        trial["income_k"] = float(income)
        if model.predict_proba(trial)[0, 1] < CUTOFF:
            return income
    return None
```

</details>

Good counterfactuals use features a person can change (income, debt). Never suggest changing age or gender, and check the suggestion is realistic.

## Present your work

Write a model-risk memo (one page):

1. **Purpose and data**: what the model decides, the base default rate.
2. **Performance**: AUC and calibration (Brier, ECE) — is the probability trustworthy?
3. **Policy**: the cut-off, approval rate, expected default rate among approved applicants.
4. **Fairness audit**: approval rates by gender, disparate impact, equal opportunity gap, and the proxy explanation.
5. **Explainability**: example reason codes and a counterfactual.
6. **Recommendation and monitoring**: what you'd change before launch, and which fairness and calibration metrics to track monthly.

## Put it on your resume

Write bullets with **your own numbers** from the milestones above. A template:

- Built an explainable default-risk model (logistic regression, test AUC **0.XX**, ECE **0.0X**) for 6,000 loan applicants
- Audited the approval policy for fairness (disparate impact **0.XX**, equal-opportunity gap **X pp**) and traced the gap to an income proxy
- Generated reason codes and income counterfactuals for rejected applicants to meet explainability requirements

Only claim what you built and measured. Put the notebook on your GitHub with a short README: problem, data, approach, results, what you'd do next.

## Interview questions about this project

Practise answering these out loud in 1–2 minutes each:

1. You removed gender from the features. Why is the model still not necessarily fair?
2. What's the difference between demographic parity and equal opportunity? Which would you use for lending?
3. Why does calibration matter for a credit model?
4. How did you generate reason codes, and when would that method be misleading?
5. The business wants a higher approval rate. What do you change, and what are the risks?
6. What would you monitor after launch?

## Take it further

- Try a gradient-boosting model, then explain it with SHAP. Is the accuracy gain worth the explainability cost?
- Find a cut-off per group that equalises the equal-opportunity gap, and discuss whether that's allowed.
- Recalibrate with `CalibratedClassifierCV` and compare ECE.
- Write a model card for this model.